In [ ]:
from typing import List

import os
import subprocess
import pickle
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt

#### Utilities

In [ ]:
def generate_ba_graph_nx(n0: int, n: int, m: int) -> nx.Graph | None:
    tmp_filename = "temporary-graph.txt"

    try:
        result = subprocess.run(
            ["../build/generate_graph", str(2), str(n0), str(m), str(n), tmp_filename],
            check=True,
        )
    except subprocess.CalledProcessError as e:
        print(f"Error generating graph: {e}")
        os.remove(tmp_filename) if os.path.exists(tmp_filename) else None
        return None

    if result.returncode != 0:
        print(f"Graph generation failed with return code {result.returncode}")
        os.remove(tmp_filename) if os.path.exists(tmp_filename) else None
        return None
    
    try:
        with open(tmp_filename, "r") as f:
            graph_data = f.read()
    except Exception as e:
        print(f"Error reading file {tmp_filename}: {e}")
        os.remove(tmp_filename) if os.path.exists(tmp_filename) else None
        return None

    graph = nx.Graph()
    for line in graph_data.splitlines():
        node1, node2 = [int(node) for node in line.strip().split()]
        graph.add_edge(node1, node2)

    return graph

In [ ]:
def save_graph_nx(graph: nx.Graph, filename: str) -> bool:
    try:
        with open(filename, "w") as f:
            for edge in graph.edges():
                f.write(f"{edge[0] + 1} {edge[1] + 1}\n")
        return True
    except Exception as e:
        print(f"Error saving graph to {filename}: {e}")
        return False


In [ ]:
def count_subgraphs(graph: nx.Graph, subgraph: nx.Graph, nthreads: int) -> int:
    save_graph_nx(graph, "graph.txt")
    save_graph_nx(subgraph, "subgraph.txt")

    try:
        result = subprocess.run(
            ["../build/_deps/peregrine-src/bin/count", "graph.txt", "subgraph.txt", str(nthreads)],
            check=True,
            text=True,
            capture_output=True,
        )

        os.remove("graph.txt")
        os.remove("subgraph.txt")

        return int(result.stdout.split(":")[2].strip())
    except Exception as e:
        os.remove("graph.txt")
        os.remove("subgraph.txt")

        print(f"Error occurred while counting subgraphs {e}")
        return -1

In [ ]:
def generate_subdivisions(graph: nx.Graph, n_max: int) -> List[nx.Graph]:
    ## iterate over all n node graphs and check if they are subdivisions of graph 
    subdivisions = []
    for n in range(1, n_max + 1):
        try:
            with open(f"graphs_n{n}.pkl", "rb") as f:
                graphs_n = pickle.load(f)
        except Exception as e:
            print(f"Error loading graphs_n{n}.pkl: {e}")
            continue

        for g in graphs_n:
            if nx.is_isomorphic(g, graph):
                continue
            if nx.algorithms.minors.is_minor(graph, g):
                subdivisions.append(g)